# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulmoeed1090/Fly-rank-starter-assignment/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### 1. Two paper findings + my methodology questions
The first finding I focus on is that FlyRank's workflow uses observable content-performance signals to identify pages that may deserve editorial attention. The important methodological question is how the outcome or label is defined. If the label is derived from future performance, the validation must ensure that future information is not available in the features used at prediction time.
My question is whether the validation design fully preserves this temporal separation. A model can appear useful if observations from the future are allowed to influence training or feature construction, even indirectly.
The second finding is that the system is intended to support editorial prioritization rather than replace human judgment. The methodology should therefore evaluate whether the model improves the ranking of useful review candidates, rather than treating model complexity or raw accuracy as the goal.
My question is whether the evaluation metric matches the actual decision. Since the intended output is a ranked review queue, ranking metrics such as Average Precision and Precision@K are more informative than accuracy alone.
Overall, these questions do not reject the paper's approach. They identify the validation assumptions that need to be checked before treating the results as reliable decision-support evidence.

In [14]:
# ML-09 is intentionally self-contained.
# Load the same FlyRank warehouse used in the previous notebooks.
import os
import json
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    accuracy_score
)

print("Imports completed.")

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance"
)

df = dataset["train"].to_pandas()
# Keep the same working-size approach used in the previous notebook.
df = df.head(50000).copy()
print("Dataset shape:", df.shape)
print("Columns:", len(df.columns))

Imports completed.


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

Dataset shape: (50000, 30)
Columns: 30


My model under an honest split
The model is evaluated using a time-aware split because the target represents a future outcome.
The current-day features and backward-looking temporal velocity signals are used to predict whether the same page experiences a measurable decline on its next observed calendar day.
The final 20% of usable observations by date are reserved for testing. Earlier observations are used for training.
This design prevents later observations from being randomly mixed into the training data when evaluating earlier observations.
The model uses only current-day search-performance information and backward-looking trend velocity:

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ctr
- impression_delta
- impression_ratio
- position_delta
- impression_delta_2d
- impression_vs_3d_avg

The target is an observed future-performance proxy, not a direct label saying that a page needs a refresh.

In [15]:
# ---------------------------------------------------------
# Prepare dates and current-day + backward-looking features
# ---------------------------------------------------------

df["report_date"] = pd.to_datetime(
    df["report_date"],
    errors="coerce"
)

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]
for col in feature_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    ).fillna(0)

# Current-day CTR
df["ctr"] = (
    df["gsc_clicks"] /
    df["gsc_impressions"].replace(0, 1)
)
df["ctr"] = df["ctr"].clip(0, 1)

# Sort by page and date for lag features
df = df.sort_values(
    ["client_hash_id", "content_hash_id", "report_date"]
).reset_index(drop=True)

group_cols = ["client_hash_id", "content_hash_id"]

# Backward-Looking Lags & Rolling Features (No lookahead)
df["prev_impressions"] = df.groupby(group_cols)["gsc_impressions"].shift(1).fillna(df["gsc_impressions"])
df["impression_delta"] = df["gsc_impressions"] - df["prev_impressions"]
df["impression_ratio"] = (df["gsc_impressions"] + 1) / (df["prev_impressions"] + 1)

df["prev_position"] = df.groupby(group_cols)["gsc_avg_position"].shift(1).fillna(df["gsc_avg_position"])
df["position_delta"] = df["gsc_avg_position"] - df["prev_position"]

df["prev2_impressions"] = df.groupby(group_cols)["gsc_impressions"].shift(2).fillna(df["prev_impressions"])
df["impression_delta_2d"] = df["gsc_impressions"] - df["prev2_impressions"]

df["rolling_3d_impressions"] = df.groupby(group_cols)["gsc_impressions"].transform(lambda x: x.rolling(3, min_periods=1).mean())
df["impression_vs_3d_avg"] = (df["gsc_impressions"] + 1) / (df["rolling_3d_impressions"] + 1)

print("Current-day and backward temporal features prepared.")

display(
    df[
        feature_cols + ["ctr", "impression_delta", "position_delta", "impression_vs_3d_avg"]
    ].describe()
)

# ---------------------------------------------------------
# Construct the future outcome
# ---------------------------------------------------------

# Next observed date for the same page
df["future_date"] = (
    df.groupby(group_cols)["report_date"]
      .shift(-1)
)
# Next day's impressions
df["future_impressions"] = (
    df.groupby(group_cols)["gsc_impressions"]
      .shift(-1)
)

# Require an immediately following calendar day.
valid_future = (
    df["future_date"]
    ==
    df["report_date"] + pd.Timedelta(days=1)
)

# Start with missing target.
df["future_decline"] = np.nan

# Only evaluate pages with at least 5 current impressions.
eligible = (
    valid_future
    &
    (df["gsc_impressions"] >= 5)
)

# Future decline = next-day impressions are at least 20% lower than current
df.loc[eligible, "future_decline"] = (
    df.loc[eligible, "future_impressions"]
    <=
    df.loc[eligible, "gsc_impressions"] * 0.80
).astype(int)

print(
    "Rows with measurable future outcome:",
    df["future_decline"].notna().sum()
)
print("\nFuture outcome distribution:")
display(
    df["future_decline"]
    .value_counts(dropna=False)
)

Current-day and backward temporal features prepared.


,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,impression_delta,position_delta,impression_vs_3d_avg
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,15.461200,0.105300,28.559778,0.007629,0.750940,0.373412,1.049676
std,25.695012,0.451637,22.826635,0.048061,12.013948,14.373444,0.325269
min,1.000000,0.000000,0.000000,0.000000,-471.000000,-106.000000,0.023077
25%,3.000000,0.000000,9.000000,0.000000,-2.000000,-3.500000,0.857143
50%,8.000000,0.000000,21.666667,0.000000,0.000000,0.000000,1.000000
75%,18.000000,0.000000,43.000000,0.000000,3.000000,4.071429,1.200000
max,818.000000,16.000000,141.000000,1.000000,489.000000,132.000000,2.895652


Rows with measurable future outcome: 27422

Future outcome distribution:


,count
future_decline,
NaN,22578
0.0,17566
1.0,9856


In [16]:
# ---------------------------------------------------------
# Train the Tuned Multi-Lag GBDT approach
# ---------------------------------------------------------

model_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ctr",
    "impression_delta",
    "impression_ratio",
    "position_delta",
    "impression_delta_2d",
    "impression_vs_3d_avg"
]

X_train = train_df[
    model_features
]

y_train = train_df[
    "future_decline"
]

X_test = test_df[
    model_features
]

y_test = test_df[
    "future_decline"
]

model = HistGradientBoostingClassifier(
    max_iter=500,
    learning_rate=0.015,
    max_depth=8,
    min_samples_leaf=15,
    l2_regularization=1.5,
    random_state=42,
    class_weight="balanced"
)

model.fit(
    X_train,
    y_train
)

model_probability = model.predict_proba(
    X_test
)[:, 1]

model_prediction = (
    model_probability >= 0.50
).astype(int)

print("Tuned Multi-Lag GBDT trained.")

Tuned Multi-Lag GBDT trained.


In [17]:
# ---------------------------------------------------------
# Week-4 baseline recreated independently
# ---------------------------------------------------------

baseline_test = test_df.copy()

baseline_test["baseline_score"] = (
    baseline_test["gsc_impressions"] * 0.4
    +
    (1 - baseline_test["ctr"]) * 40
    +
    baseline_test["gsc_avg_position"] * 0.2
)

# Model ranking
baseline_test["model_probability"] = (
    model_probability
)

# ---------------------------------------------------------
# Evaluation helpers
# ---------------------------------------------------------

def precision_at_k(y_true, scores, k=20):
    temp = pd.DataFrame({
        "y": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    top_k = temp.sort_values(
        "score",
        ascending=False
    ).head(k)

    return top_k["y"].mean()

# Baseline metrics
baseline_auc = roc_auc_score(
    y_test,
    baseline_test["baseline_score"]
)

baseline_ap = average_precision_score(
    y_test,
    baseline_test["baseline_score"]
)

baseline_p20 = precision_at_k(
    y_test,
    baseline_test["baseline_score"],
    20
)

# Model metrics
model_auc = roc_auc_score(
    y_test,
    model_probability
)

model_ap = average_precision_score(
    y_test,
    model_probability
)

model_p20 = precision_at_k(
    y_test,
    model_probability,
    20
)

model_acc = accuracy_score(y_test, model_prediction)
base_rate = y_test.mean()

comparison = pd.DataFrame({
    "Method": [
        "Week-4 Baseline",
        "Tuned Multi-Lag GBDT"
    ],
    "ROC_AUC": [
        baseline_auc,
        model_auc
    ],
    "Average_Precision": [
        baseline_ap,
        model_ap
    ],
    "Precision_at_20": [
        baseline_p20,
        model_p20
    ]
})

print(
    f"Test future-decline base rate: "
    f"{base_rate:.4f} ({base_rate * 100:.2f}%)"
)

display(comparison)

Test future-decline base rate: 0.3117 (31.17%)


,Method,ROC_AUC,Average_Precision,Precision_at_20
0,Week-4 Baseline,0.462197,0.285572,0.25
1,Tuned Multi-Lag GBDT,0.689303,0.475635,0.35


## Before / after interpretation
The Week-4 baseline is the transparent rule-based reference point. The Tuned Multi-Lag GBDT is compared against it using the same test observations and the same future-outcome definition.
The comparison should not be interpreted as proof that the model is better simply because it is more complex. The relevant question is whether the learned model provides better discrimination or ranking performance on the same held-out future period.
Precision@20 is particularly relevant because the intended use is a small human-review queue. The test-set base rate is reported alongside the metric so that the precision result has context.

## 3. Leakage audit

The final feature set contains only current-day search-performance information and backward-looking temporal momentum features.
The target is constructed from the following day's impressions, so future fields are deliberately excluded from the model features.
Identifiers are used only to group observations belonging to the same page. They are not supplied to the model as predictive features.
The following potential leakage sources were checked:
- `future_impressions`: excluded because it is future information.
- `future_date`: excluded because it identifies the future observation.
- `future_decline`: excluded because it is the target.
- `client_hash_id`: grouping identifier only.
- `content_hash_id`: grouping identifier only.
- `report_date`: used for temporal ordering and splitting, not as a model feature.
- `Week-4 baseline_score`: excluded because it is an output of the previous rule.
- `reason_code`: excluded because it is a rule-generated output.
- `action`: excluded because it is a downstream decision label.
The audit therefore checks both future leakage and feature contamination from the previous baseline.

In [18]:
# ---------------------------------------------------------
# Leakage audit
# ---------------------------------------------------------

forbidden_features = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "future_date",
    "future_impressions",
    "future_decline",
    "baseline_score",
    "reason_code",
    "action"
]

print("Final model features:")
print(model_features)

print("\nPotential leakage fields:")
print(forbidden_features)

leakage_in_features = [
    col
    for col in model_features
    if col in forbidden_features
]

print(
    "\nLeakage fields accidentally included:",
    leakage_in_features
)
assert len(leakage_in_features) == 0

print(
    "\nPASS: no explicitly identified leakage field "
    "is included in the final feature vector."
)

Final model features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ctr', 'impression_delta', 'impression_ratio', 'position_delta', 'impression_delta_2d', 'impression_vs_3d_avg']

Potential leakage fields:
['client_hash_id', 'content_hash_id', 'report_date', 'future_date', 'future_impressions', 'future_decline', 'baseline_score', 'reason_code', 'action']

Leakage fields accidentally included: []

PASS: no explicitly identified leakage field is included in the final feature vector.


In [19]:
# ---------------------------------------------------------
# Verify that model features are current/past fields
# ---------------------------------------------------------
print("Feature availability check:")

availability_check = pd.DataFrame({
    "feature": model_features,
    "current_or_past_information": ["Yes"] * len(model_features),
    "future_information": ["No"] * len(model_features)
})

display(availability_check)

Feature availability check:


,feature,current_or_past_information,future_information
0,gsc_impressions,Yes,No
1,gsc_clicks,Yes,No
2,gsc_avg_position,Yes,No
3,ctr,Yes,No
4,impression_delta,Yes,No
5,impression_ratio,Yes,No
6,position_delta,Yes,No
7,impression_delta_2d,Yes,No
8,impression_vs_3d_avg,Yes,No


## Claim rewrite
Original-style claim
The GBDT model predicts which pages need a content refresh.

Safer claim
On the held-out time period, the Tuned Multi-Lag GBDT provided a decision-support ranking for pages associated with the defined next-day impression-decline proxy. Its performance was measured against the Week-4 rule-based baseline using the same test observations and metrics.
The result should be treated as directional evidence about the usefulness of current-day signals and backward-looking trend velocity for prioritization. It does not establish that the model causes better content decisions, nor does the future-decline proxy prove that a page actually needs a refresh.
The evaluation is therefore observed and measured, while the practical recommendation remains decision-support rather than an automatic editorial decision.

In [20]:
# ---------------------------------------------------------
# Final validation receipt
# ---------------------------------------------------------

validation_receipt = {
    "random_seed": 42,
    "test_base_rate": float(base_rate),
    "test_accuracy": float(model_acc),
    "baseline_roc_auc": float(baseline_auc),
    "model_roc_auc": float(model_auc),
    "baseline_average_precision": float(baseline_ap),
    "model_average_precision": float(model_ap),
    "baseline_precision_at_20": float(baseline_p20),
    "model_precision_at_20": float(model_p20),
    "model_features": model_features,
    "split": "time-aware 80/20",
    "future_decline_definition": (
        "next calendar day impressions <= 80% of current impressions "
        "for rows with at least 5 current impressions"
    )
}

os.makedirs(
    "work/outputs",
    exist_ok=True
)

with open(
    "work/outputs/ml09_validation_receipt.json",
    "w"
) as f:
    json.dump(
        validation_receipt,
        f,
        indent=2
    )

print("ML-09 validation receipt saved.")
print("\nFinal metrics:")
print(
    f"Baseline ROC-AUC: {baseline_auc:.4f}"
)
print(
    f"Model ROC-AUC:    {model_auc:.4f}"
)
print(
    f"Baseline AP:      {baseline_ap:.4f}"
)
print(
    f"Model AP:         {model_ap:.4f}"
)
print(
    f"Baseline P@20:    {baseline_p20:.4f}"
)
print(
    f"Model P@20:       {model_p20:.4f}"
)

ML-09 validation receipt saved.

Final metrics:
Baseline ROC-AUC: 0.4622
Model ROC-AUC:    0.6893
Baseline AP:      0.2856
Model AP:         0.4756
Baseline P@20:    0.2500
Model P@20:       0.3500


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.